# pypgo.implicit API Demo

This notebook introduces the lazy implicit surface API:
`GridSpec`, analytic fields, CSG operators, grid sampling,
marching-cubes extraction, and mesh surface thickening.


In [1]:
import numpy as np
import pypgo as pgo
from pypgo import implicit


## 1. Analytic fields

`SphereField` and `BoxField` are `ImplicitField` objects. They can be
queried at points without allocating a sampled grid.


In [2]:
sphere = implicit.SphereField([0.0, 0.0, 0.0], 1.0)
box = implicit.BoxField([0.0, 0.0, 0.0], [0.75, 0.75, 0.75])

points = np.array(
    [
        [0.0, 0.0, 0.0],
        [1.0, 0.0, 0.0],
        [1.5, 0.0, 0.0],
    ],
    dtype=np.float64,
)

print("sphere eval:", [sphere.eval(p) for p in points])
print("box eval:", [box.eval(p) for p in points])
print("sphere bounds:", sphere.bounds())


sphere eval: [-1.0, 0.0, 0.5]
box eval: [-0.75, 0.25, 0.75]
sphere bounds: (array([-1., -1., -1.]), array([1., 1., 1.]))


## 2. Lazy CSG

The `|`, `&`, and `-` operators create lazy composed fields. No dense
grid is allocated until `sample_to_grid` is called.


In [3]:
left = implicit.SphereField([-0.45, 0.0, 0.0], 0.9)
right = implicit.SphereField([0.45, 0.0, 0.0], 0.9)

union = left | right
intersection = left & right
difference = left - right

p = np.array([0.0, 0.0, 0.0], dtype=np.float64)
print("union at origin:", union.eval(p))
print("intersection at origin:", intersection.eval(p))
print("difference at origin:", difference.eval(p))
print("lazy type:", type(union).__name__)


union at origin: -0.45
intersection at origin: -0.45
difference at origin: 0.45
lazy type: ImplicitField


## 3. Sampling to `GridField`

`sample_to_grid` materializes a field on a uniform grid. `GridField`
still inherits `ImplicitField`, so point queries use trilinear
interpolation and CSG operators continue to work.


In [4]:
spec = implicit.GridSpec([-1.5, -1.5, -1.5], [1.5, 1.5, 1.5], resolution=48)
grid = union.sample_to_grid(spec, num_threads=1)

values = grid.values
print("grid shape:", values.shape)
print("grid dtype:", values.dtype)
print("min/max:", float(values.min()), float(values.max()))
print("grid eval at origin:", grid.eval([0.0, 0.0, 0.0]))
print("values shares memory:", np.shares_memory(values, grid.values))


grid shape: (48, 48, 48)
grid dtype: float64
min/max: -0.8465008895290207 1.4669600757089252
grid eval at origin: -0.4794856993532347
values shares memory: True


## 4. Marching cubes

Extraction accepts a materialized `GridField`. For an unsampled lazy
field, call `sample_to_grid` first.


In [5]:
surface = implicit.extract_marching_cubes(grid, iso_offset=0.0)
print("vertices:", surface.num_vertices)
print("triangles:", surface.num_elements)
print("bbox:", surface.bbox)


vertices: 5528
triangles: 11052
bbox: (array([-1.3488662 , -0.89897393, -0.89897393]), array([1.3488662 , 0.89897393, 0.89897393]))


## 5. Mesh unsigned distance and shell thickening

`MeshUnsignedDistanceField` turns a triangle surface into a distance
field. The convenience function below performs the common pipeline:
mesh -> distance field -> offset -> grid -> marching cubes.


In [6]:
tri = pgo.mesh.TriMeshData(
    [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]],
    [[0, 1, 2]],
)

shell = implicit.thicken_mesh_surface(
    tri,
    thickness=0.1,
    resolution=24,
    padding=0.25,
)

print("shell vertices:", shell.num_vertices)
print("shell triangles:", shell.num_elements)


shell vertices: 558
shell triangles: 1112


## 6. OpenVDB availability

OpenVDB support depends on how libpgo was built. Use `has_openvdb`
before constructing OpenVDB level sets.


In [7]:
print("OpenVDB available:", implicit.has_openvdb())

if implicit.has_openvdb():
    opts = implicit.OpenVDBOptions(voxel_size=0.05)
    levelset = implicit.build_openvdb_from_grid_field(grid, opts)
    vdb_surface = implicit.extract_openvdb(levelset, opts)
    print("OpenVDB surface:", vdb_surface.num_vertices, vdb_surface.num_elements)


OpenVDB available: True
OpenVDB surface: 5530 11056
